# 04 Backtest

Load trained model artifacts, generate per-model and ensemble probabilities on the chronological test split, run the intraday backtest, and write reports including monthly Sharpe and win rate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

%cd $REPO_DIR
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
import json

import pandas as pd
import torch

from backtest.engine import BacktestEngine
from backtest.metrics import compute_metrics, monthly_breakdown
from config import BacktestConfig, FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig, SignalConfig, SplitConfig
from features.pipeline import FeatureEngineeringPipeline
from features.sequences import SequenceBuilder
from models.lgbm import LightGBMModel
from models.splits import chronological_split
from models.torch_models import GRUClassifier, LSTMClassifier
from models.torch_training import predict_torch_probabilities
from reports import write_backtest_reports
from strategy.signals import SignalFuser

paths = PathConfig(root=BASE, raw_data_dir=DATA_DIR / 'raw', processed_data_dir=DATA_DIR / 'processed', artifact_dir=ARTIFACT_DIR, model_artifact_dir=MODEL_DIR, report_dir=REPORT_DIR)
if (DATA_DIR / 'processed' / 'BANKNIFTY_5m_features.parquet').exists():
    market = MarketConfig(symbol='BANKNIFTY', ticker='BANKNIFTY', series='EQ', interval='5m', intraday_source='drive-cache-local-csv', daily_source='intraday-resample')
elif (DATA_DIR / 'processed' / 'NIFTY_5m_features.parquet').exists():
    market = MarketConfig(symbol='NIFTY', ticker='NIFTY', series='EQ', interval='5m', intraday_source='drive-cache-local-csv', daily_source='intraday-resample')
else:
    market = MarketConfig(symbol='RELIANCE', ticker='RELIANCE.NS', series='EQ', interval='5m', intraday_source='drive-cache-or-jugaad-openchart')
labels = LabelConfig(horizon=15, target_pct=0.005, stop_pct=0.003)
pipeline = FeatureEngineeringPipeline(paths=paths, market=market, features=FeatureConfig(), labels=labels, normalizer=NormalizerConfig())
dataset = pipeline.load()
splits = chronological_split(dataset.frame, SplitConfig())
test_frame = splits.test
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Test rows={len(test_frame):,} device={device}')

In [ ]:
probabilities = {}

lgbm_runner = LightGBMModel(paths=paths, market=market)
lgbm_model = lgbm_runner.load()
probabilities['lgbm'] = lgbm_runner.predict_probabilities(lgbm_model, test_frame, dataset.feature_columns)

metadata_path = MODEL_DIR / 'torch_training_metadata.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    lookback = int(metadata['lookback'])
    input_size = int(metadata['input_size'])
    test_arrays = SequenceBuilder(lookback=lookback).build(test_frame, feature_columns=dataset.feature_columns)

    lstm_path = MODEL_DIR / 'lstm_final.pt'
    if lstm_path.exists():
        lstm = LSTMClassifier(input_size=input_size)
        lstm.load_state_dict(torch.load(lstm_path, map_location=device)['model_state'])
        probabilities['lstm'] = predict_torch_probabilities(model=lstm, arrays=test_arrays, device=device)

    gru_path = MODEL_DIR / 'gru_final.pt'
    if gru_path.exists():
        gru = GRUClassifier(input_size=input_size)
        gru.load_state_dict(torch.load(gru_path, map_location=device)['model_state'])
        probabilities['gru'] = predict_torch_probabilities(model=gru, arrays=test_arrays, device=device)

print('Models available:', sorted(probabilities))

In [ ]:
def aligned_average_probability(probability_map):
    common_index = None
    for frame in probability_map.values():
        common_index = frame.index if common_index is None else common_index.intersection(frame.index)
    aligned = [frame.loc[common_index, ['p_sell', 'p_hold', 'p_buy']] for frame in probability_map.values()]
    return sum(aligned) / len(aligned)

ensemble_probabilities = aligned_average_probability(probabilities)
ensemble_frame = test_frame.loc[ensemble_probabilities.index]
volume_multiplier = 0.0 if 'local' in market.intraday_source else 1.5
signals = SignalFuser(SignalConfig(confidence_threshold=0.65, volume_multiplier=volume_multiplier)).generate(ensemble_frame, ensemble_probabilities)
result = BacktestEngine(config=BacktestConfig(), labels=labels).run(signals)
metrics = compute_metrics(result.trades, result.equity_curve, BacktestConfig())
monthly = monthly_breakdown(result.trades)

output_paths = write_backtest_reports(report_dir=REPORT_DIR, signals=signals, trades=result.trades, equity_curve=result.equity_curve, metrics=metrics, monthly=monthly)
print(metrics)
print(monthly.tail())
print(output_paths)